# ESSE 4360 — AI for Payloads · Notebook 1
## Classifying Sentinel-2 scenes (EuroSAT RGB): from simple features to a CNN

**What you will do**
1. Download **EuroSAT**: 27 000 Sentinel-2 image tiles (64 × 64 px, 10 m pixels) in 10 land-cover classes (Helber et al., 2019; MIT licence).
2. Train **classical** classifiers on simple colour features: logistic regression, random forest, and k-means (no labels).
3. Train a **small convolutional neural network** (about 24 000 parameters) that runs on a laptop CPU.
4. Optional, on a GPU: **fine-tune a pre-trained ResNet-18** (transfer learning).
5. Score every model with accuracy, per-class precision/recall and a confusion matrix — and look at the mistakes.

**Runs on:** Google Colab (free tier) or any laptop with Python. CPU is fine with the default settings (about 5–10 minutes in total). For a GPU in Colab: *Runtime → Change runtime type → T4 GPU*.

All figures are saved in a `figures/` folder; the last cell zips them for download.

In [ ]:
# Setup (Colab already has these packages; on a laptop: pip install torch torchvision scikit-learn matplotlib)
import os, time, random, zipfile
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torchvision import datasets
from PIL import Image

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Running on:', device)

DATA, FIG = 'data', 'figures'
os.makedirs(DATA, exist_ok=True); os.makedirs(FIG, exist_ok=True)

# Size knobs: increase on a GPU, decrease on a slow laptop
N_PER_CLASS = 1000        # images per class to load (max ~2000-3000; 1000 x 10 classes = 10 000 images)
CNN_EPOCHS = 8 if device.type == 'cpu' else 15
RUN_RESNET = device.type == 'cuda'   # transfer learning is slow on CPU; set True to force it

## 1. Download the data
`torchvision` downloads EuroSAT (about 90 MB) the first time and reuses it afterwards. Images are stored as one folder per class.

In [ ]:
ds = datasets.EuroSAT(root=DATA, download=True)
CLASSES = ds.classes
print(len(ds), 'images in', len(CLASSES), 'classes:', CLASSES)

# Load a balanced subset into memory as a NumPy array: (N, 64, 64, 3), values 0-255
by_class = {c: [] for c in range(len(CLASSES))}
for path, c in ds.samples:
    by_class[c].append(path)
paths, labels = [], []
for c, plist in by_class.items():
    random.shuffle(plist)
    paths += plist[:N_PER_CLASS]; labels += [c] * len(plist[:N_PER_CLASS])
X = np.stack([np.asarray(Image.open(p).convert('RGB')) for p in paths])
y = np.array(labels)
print('Loaded', X.shape, 'labels', np.bincount(y))

## 2. Look at the data first
Always look before you train. Which classes look alike? Which will be easy?

In [ ]:
fig, ax = plt.subplots(len(CLASSES), 8, figsize=(10, 13))
for c in range(len(CLASSES)):
    idx = np.random.choice(np.flatnonzero(y == c), 8, replace=False)
    for k, i in enumerate(idx):
        ax[c, k].imshow(X[i]); ax[c, k].axis('off')
    ax[c, 0].text(-8, 32, CLASSES[c], ha='right', va='center', fontsize=10)
plt.suptitle('EuroSAT examples (Sentinel-2, 64 x 64 px at 10 m)', y=0.995)
plt.tight_layout(); plt.savefig(f'{FIG}/01_examples.png', dpi=150); plt.show()

## 3. Split the data: train / validation / test
We use a stratified random split (70 / 15 / 15 %).

> **Think:** EuroSAT tiles were cut from many Sentinel-2 scenes, and neighbouring tiles can be very similar. A random split can therefore be optimistic. A stricter test holds out whole regions (TorchGeo provides a spatial split). Keep this in mind when you read the scores.

In [ ]:
from sklearn.model_selection import train_test_split
idx_all = np.arange(len(y))
i_tr, i_tmp = train_test_split(idx_all, test_size=0.30, stratify=y, random_state=SEED)
i_va, i_te = train_test_split(i_tmp, test_size=0.50, stratify=y[i_tmp], random_state=SEED)
print('train', len(i_tr), ' validation', len(i_va), ' test', len(i_te))

## 4. Classical machine learning on hand-made features
A person chooses the features. Here: the mean and standard deviation of each colour band, plus an 8-bin brightness histogram per band (30 numbers per image).

In [ ]:
def colour_features(imgs):
    f = [imgs.mean((1, 2)), imgs.std((1, 2))]
    for b in range(3):
        h = np.stack([np.histogram(im[..., b], bins=8, range=(0, 255))[0] for im in imgs]) / (64 * 64)
        f.append(h)
    return np.concatenate(f, 1)

F_all = colour_features(X.astype(np.float32))
print('feature matrix:', F_all.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

results = {}
models = {'Logistic regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
          'Random forest': RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED)}
for name, m in models.items():
    t0 = time.time(); m.fit(F_all[i_tr], y[i_tr])
    acc = accuracy_score(y[i_te], m.predict(F_all[i_te])); results[name] = acc
    print(f'{name:20s} test accuracy {acc * 100:.1f} %   ({time.time() - t0:.1f} s)')

### Unsupervised: k-means on the same features (no labels used)
k-means finds 10 groups. To score it, we match each cluster to the class it overlaps most (Hungarian matching).

In [ ]:
from sklearn.cluster import KMeans
from scipy.optimize import linear_sum_assignment
Fs = StandardScaler().fit_transform(F_all)
km = KMeans(n_clusters=len(CLASSES), n_init=10, random_state=SEED).fit(Fs[i_tr])
cl_te = km.predict(Fs[i_te])
C = confusion_matrix(y[i_te], cl_te, labels=range(len(CLASSES)))
r, c = linear_sum_assignment(-C)
mapping = dict(zip(c, r))
acc_km = np.mean([mapping[k] == t for k, t in zip(cl_te, y[i_te])]); results['k-means (no labels)'] = acc_km
print(f'k-means after matching clusters to classes: {acc_km * 100:.1f} %')

## 5. A small convolutional neural network
The CNN learns its own features from the pixels. Three blocks of *convolution → batch normalisation → ReLU → 2 × 2 max-pooling* shrink the 64 × 64 image to 8 × 8, then global average pooling and one linear layer give 10 class scores.

In [ ]:
# ---------- a small, reusable training loop (used by every network below) ----------
def augment(x):
    """Random flips and 90-degree rotations: valid for nadir-looking imagery."""
    if torch.rand(1) < 0.5: x = x.flip(-1)
    if torch.rand(1) < 0.5: x = x.flip(-2)
    return torch.rot90(x, int(torch.randint(0, 4, (1,))), (-2, -1))


def train_model(model, Xtr, ytr, Xva, yva, epochs=8, lr=1e-3, bs=64, aug=True):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss()
    hist = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    Xtr_t, ytr_t = torch.as_tensor(Xtr), torch.as_tensor(ytr)
    for ep in range(epochs):
        model.train(); t0 = time.time(); perm = torch.randperm(len(Xtr_t)); tot = 0
        for i in range(0, len(perm), bs):
            idx = perm[i:i + bs]
            xb = Xtr_t[idx].to(device); yb = ytr_t[idx].to(device)
            if aug: xb = augment(xb)
            opt.zero_grad(); loss = lossf(model(xb), yb); loss.backward(); opt.step()
            tot += loss.item() * len(idx)
        vl, va = evaluate(model, Xva, yva)
        hist['train_loss'].append(tot / len(perm)); hist['val_loss'].append(vl); hist['val_acc'].append(va)
        print(f'epoch {ep + 1:2d}  train loss {tot / len(perm):.3f}  val loss {vl:.3f}  val acc {va * 100:.1f} %  ({time.time() - t0:.0f} s)')
    return model, hist


@torch.no_grad()
def predict(model, X, bs=256):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        out.append(model(torch.as_tensor(X[i:i + bs]).to(device)).softmax(1).cpu())
    return torch.cat(out).numpy()


def evaluate(model, X, y):
    p = predict(model, X)
    loss = nn.functional.nll_loss(torch.log(torch.as_tensor(p) + 1e-9), torch.as_tensor(y)).item()
    return loss, (p.argmax(1) == y).mean()


def plot_history(hist, title, fname):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(hist['train_loss'], 'o-', label='train'); ax[0].plot(hist['val_loss'], 'o-', label='validation')
    ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].legend(); ax[0].set_title(title)
    ax[1].plot(np.array(hist['val_acc']) * 100, 'o-', color='C2'); ax[1].set_xlabel('epoch'); ax[1].set_ylabel('validation accuracy (%)')
    plt.tight_layout(); plt.savefig(f'{FIG}/{fname}', dpi=150); plt.show()


class SmallCNN(nn.Module):
    """Three conv blocks (conv -> batch norm -> ReLU -> 2x2 max pool), then global average pooling."""
    def __init__(self, n_in=3, n_classes=10, width=16):
        super().__init__()
        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        self.features = nn.Sequential(block(n_in, width), block(width, 2 * width), block(2 * width, 4 * width))  # 64 -> 8 px
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(4 * width, n_classes))

    def forward(self, x):
        return self.head(self.features(x))


# Scale pixels to 0-1 and arrange as (N, channels, height, width)
Xn = (X.astype(np.float32) / 255.0).transpose(0, 3, 1, 2)
cnn = SmallCNN(n_in=3, n_classes=len(CLASSES))
print('SmallCNN parameters:', sum(p.numel() for p in cnn.parameters()))

In [ ]:
cnn, hist = train_model(cnn, Xn[i_tr], y[i_tr], Xn[i_va], y[i_va], epochs=CNN_EPOCHS)
plot_history(hist, 'SmallCNN on EuroSAT RGB', '02_cnn_training.png')
p_cnn = predict(cnn, Xn[i_te]).argmax(1)
results['Small CNN'] = accuracy_score(y[i_te], p_cnn)
print(f"Small CNN test accuracy: {results['Small CNN'] * 100:.1f} %")

## 6. Look inside: what did the first layer learn?
Each first-layer kernel is a 3 × 3 × 3 filter. Many look like colour or edge detectors.

In [ ]:
w = cnn.features[0][0].weight.detach().cpu().numpy()          # (16, 3, 3, 3)
fig, ax = plt.subplots(2, 8, figsize=(10, 2.8))
for k, a in enumerate(ax.ravel()):
    k_img = w[k].transpose(1, 2, 0); k_img = (k_img - k_img.min()) / (k_img.max() - k_img.min() + 1e-9)
    a.imshow(k_img, interpolation='nearest'); a.axis('off')
plt.suptitle('Learned first-layer kernels (3 x 3, RGB)'); plt.tight_layout(); plt.savefig(f'{FIG}/03_kernels.png', dpi=150); plt.show()

## 7. Optional: transfer learning with a pre-trained ResNet-18 (GPU recommended)
Start from a network trained on 1.2 million everyday photos (ImageNet), replace the last layer, and fine-tune on EuroSAT.

In [ ]:
if RUN_RESNET:
    from torchvision import models
    import torch.nn.functional as Fn

    class ResNetEO(nn.Module):
        def __init__(self, n_classes):
            super().__init__()
            self.net = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
            self.net.fc = nn.Linear(self.net.fc.in_features, n_classes)
            self.register_buffer('mu', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
            self.register_buffer('sd', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        def forward(self, x):
            x = Fn.interpolate(x, size=128, mode='bilinear', align_corners=False)   # ResNet likes bigger inputs
            return self.net((x - self.mu) / self.sd)

    rn = ResNetEO(len(CLASSES))
    print('ResNet-18 parameters:', sum(p.numel() for p in rn.parameters()))
    rn, hist_rn = train_model(rn, Xn[i_tr], y[i_tr], Xn[i_va], y[i_va], epochs=4, lr=3e-4)
    results['ResNet-18 (pre-trained)'] = accuracy_score(y[i_te], predict(rn, Xn[i_te]).argmax(1))
    print(f"ResNet-18 test accuracy: {results['ResNet-18 (pre-trained)'] * 100:.1f} %")
else:
    print('Skipped (no GPU). Set RUN_RESNET = True in the setup cell to run it anyway.')

## 8. Compare and diagnose

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
names = list(results); vals = [results[k] * 100 for k in names]
ax.barh(names, vals, color=['0.6'] * 3 + ['C0'] * (len(names) - 3)); ax.set_xlim(0, 100); ax.set_xlabel('test accuracy (%)')
for i, v in enumerate(vals): ax.text(v + 1, i, f'{v:.1f}', va='center')
ax.invert_yaxis(); plt.tight_layout(); plt.savefig(f'{FIG}/04_comparison.png', dpi=150); plt.show()

print(classification_report(y[i_te], p_cnn, target_names=CLASSES, digits=3, zero_division=0))
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(confusion_matrix(y[i_te], p_cnn), display_labels=CLASSES).plot(ax=ax, xticks_rotation=60, colorbar=False, cmap='Blues')
ax.set_title('Small CNN confusion matrix (test set)'); plt.tight_layout(); plt.savefig(f'{FIG}/05_confusion.png', dpi=150); plt.show()

In [ ]:
# The mistakes are the most instructive part
wrong = np.flatnonzero(p_cnn != y[i_te])
pick = np.random.choice(wrong, min(12, len(wrong)), replace=False)
fig, ax = plt.subplots(2, 6, figsize=(13, 5))
for a, k in zip(ax.ravel(), pick):
    a.imshow(X[i_te[k]]); a.axis('off')
    a.set_title(f'true: {CLASSES[y[i_te[k]]]}\npred: {CLASSES[p_cnn[k]]}', fontsize=9, color='firebrick')
plt.suptitle('Misclassified test tiles'); plt.tight_layout(); plt.savefig(f'{FIG}/06_mistakes.png', dpi=150); plt.show()

## 9. Questions
1. Which classes does the CNN confuse most? Look at the mistakes: is the label even clear to *you*?
2. The random forest uses only colour. Which classes need *texture or shape* that only the CNN can see?
3. How far below the supervised models is k-means? What does that say about the value of labels?
4. Change `N_PER_CLASS` to 200. How much does each model lose? Which one degrades most gracefully?
5. **Payload question:** our CNN shrinks 64 × 64 to 8 × 8 before deciding. Would it still find a 2-pixel ship? What would you change?

In [ ]:
# Download all figures (Colab) or find them in the figures/ folder (laptop)
with zipfile.ZipFile('figures_notebook1.zip', 'w') as z:
    for f in sorted(os.listdir(FIG)): z.write(os.path.join(FIG, f))
try:
    from google.colab import files; files.download('figures_notebook1.zip')
except ImportError:
    print('Saved figures_notebook1.zip')